In [1]:
import os, requests

def download_xray_density(pdb_id, cache_dir="./edensity_cache"):
    """Download 2Fo-Fc map from PDBe. Returns path or None."""
    pdb_id = pdb_id.lower()
    out_path = os.path.join(cache_dir, f"{pdb_id}.ccp4")
    if os.path.exists(out_path):
        return out_path
    
    url = f"https://www.ebi.ac.uk/pdbe/entry-files/{pdb_id}.ccp4"
    resp = requests.get(url, timeout=60)
    if resp.status_code == 200:
        os.makedirs(cache_dir, exist_ok=True)
        with open(out_path, "wb") as f:
            f.write(resp.content)
        return out_path
    return None  # no map available → use Gaussian fallback

# Batch download for all unique PDBs in CrossDocked2020:
unique_pdbs = extract_unique_pdbs_from_crossdocked()  # ~3,000
results = {pdb: download_xray_density(pdb) for pdb in unique_pdbs}
available = sum(1 for v in results.values() if v is not None)
print(f"Maps available: {available}/{len(unique_pdbs)}")

NameError: name 'extract_unique_pdbs_from_crossdocked' is not defined

In [ ]:
# Expected output: Maps available: ~2600/3000
```

### Storage & Time Estimates

| What | Estimate |
|---|---|
| Unique PDB IDs to download | ~3,000 |
| Map file size | 1–10 MB each |
| Total download size | ~10–30 GB |
| Download time (parallel) | 2–4 hours |
| Crop + resample per structure | ~10 ms |
| Total preprocessing time | minutes |

### Summary: Your Complete Data Pipeline
```
CrossDocked2020 filename
    │
    ├── Parse: receptor_pdb = "2z3h"
    │
    ├── Download: https://www.ebi.ac.uk/pdbe/entry-files/2z3h.ccp4
    │   └── Success (~85-90%): real X-ray 2Fo-Fc electron density
    │   └── Fail (~10-15%): Gaussian smearing fallback
    │
    ├── Crop: gemmi → pocket bounding box (23 Å, 0.5 Å grid)
    │
    ├── Normalize: sigma-scaling
    │
    └── Output: (1, 46, 46, 46) extra channel for VoxBind U-Net